# Rule of Thumb on MIT1003: saliency vs human fixations

This notebook explains a stock ImageNet ResNet18 with an image RoT
(`fit_image` over MobileNetV3-Small feature maps, plus a raw-pixel arm for
contrast) and gauges the saliency maps against human eye-tracking fixation
maps. Pixel overlap is the wrong gauge for diffuse maps, so both sides are
reduced to **bounding boxes** (top-10%-mass, connected components) and
compared by pointing-game hits, greedy IoU and mass coverage — identically
for RoT, Integrated Gradients, occlusion, a center prior and noise.

Fixations record where humans look (plausibility), not what the classifier
computed: the center prior wins, honestly reported. Everything runs on CPU;
the only downloads are the two MIT zips plus cached model weights.

In [1]:
import os
import urllib.request
import zipfile

import numpy as np

CACHE = "./_sal_cache"
os.makedirs(os.path.join(CACHE, "mit1003"), exist_ok=True)
MIT = "http://people.csail.mit.edu/tjudd/WherePeopleLook/"
for name in ("ALLSTIMULI.zip", "ALLFIXATIONMAPS.zip"):
    dest = os.path.join(CACHE, name)
    if not os.path.exists(dest):
        print(f"downloading {name} ...", flush=True)
        urllib.request.urlretrieve(MIT + name, dest)
    with zipfile.ZipFile(dest) as z:
        z.extractall(os.path.join(CACHE, "mit1003"))
print("extracted", flush=True)

extracted


In [2]:
from PIL import Image

stim, fixd = None, None
for root, _, files in os.walk(os.path.join(CACHE, "mit1003")):
    if "MACOSX" in root:
        continue
    base = os.path.basename(root).upper()
    if base == "ALLSTIMULI":
        stim = root
    elif base == "ALLFIXATIONMAPS":
        fixd = root
if stim is None or fixd is None:
    for root, _, files in os.walk(os.path.join(CACHE, "mit1003")):
        if "MACOSX" in root:
            continue
        if stim is None and any(f.lower().endswith(".jpeg") for f in files):
            stim = root
        if fixd is None and any("_fixmap" in f.lower() for f in files):
            fixd = root
assert stim is not None and fixd is not None, "stimulus/fixation dirs not found"
pairs = []
for f in sorted(os.listdir(stim)):
    stem = os.path.splitext(f)[0]
    cand = stem + "_fixMap.jpg"
    if f.lower().endswith((".jpeg", ".jpg")) and os.path.exists(os.path.join(fixd, cand)):
        pairs.append((os.path.join(stim, f), os.path.join(fixd, cand)))
sel = np.random.RandomState(0).choice(len(pairs), 500, replace=False)
Xs, Fs = [], []
for i in sel:
    p, q = pairs[i]
    Xs.append(np.array(Image.open(p).convert("RGB").resize((128, 128))))
    f = np.array(Image.open(q).convert("L").resize((128, 128))).astype(np.float64)
    Fs.append(f / f.sum())
X = np.stack(Xs)
F = np.stack(Fs).astype(np.float32)
print(f"500-set: X {X.shape} F {F.shape}", flush=True)
idx = np.random.RandomState(7).choice(500, 150, replace=False)

500-set: X (500, 128, 128, 3) F (500, 128, 128)


In [3]:
import torch
from torchvision.models import ResNet18_Weights, resnet18

bb = resnet18(weights=ResNet18_Weights.DEFAULT).eval()
with torch.no_grad():
    xb = torch.from_numpy(X.transpose(0, 3, 1, 2).astype(np.float32) / 255.0)
    P = torch.cat([bb(xb[i:i + 64]).argmax(1) for i in range(0, 500, 64)]).numpy()
print(f"black-box spread: {len(np.unique(P))} classes over 500 photos", flush=True)

black-box spread: 208 classes over 500 photos


## Fit both RoT arms

Maps arm: MobileNetV3-Small features of the 150-subset, 1000-way, 60
epochs. Pixel arm: the same 150 photos at 32px. Fidelity decides which arm
to trust — the two arms visibly disagree, and only one tracks the box.

In [4]:
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

import ruleofthumb as rot

mob_back = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT).features.eval()
with torch.no_grad():
    t = torch.from_numpy(X[idx].transpose(0, 3, 1, 2).astype(np.float32) / 255.0)
    Fm = torch.cat([mob_back(t[i:i + 32]) for i in range(0, 150, 32)]).numpy().astype(np.float32)
print(f"maps: {Fm.shape}", flush=True)
exp_mob = rot.fit_image(P[idx], Fm, epochs=60, batch_size=32, n_classes=1000, seed=0)
print(f"maps train agreement: {exp_mob.train_agreement_:.3f}", flush=True)
Xp = np.stack([np.array(Image.fromarray(X[j]).resize((32, 32))).transpose(2, 0, 1) for j in idx]).astype(np.float32) / 255.0
exp_pix = rot.fit_image(P[idx], Xp, epochs=60, batch_size=32, n_classes=1000, seed=0)
print(f"pixel train agreement: {exp_pix.train_agreement_:.3f}", flush=True)

maps: (150, 576, 4, 4)


maps train agreement: 0.980


pixel train agreement: 0.187


/Users/bigcamel/Oxford/Projects/Rule-of-Thumb-Explaining-Artificial-Intelligence-Systems-using-Partial-Information/Rule-of-Thumb/src/ruleofthumb/explain.py:84: UserWarning: low train agreement (0.187 < 0.75); explanations may be unreliable. Check surrogate-vs-blackbox agreement before trusting them (see the capacity docs).
  warnings.warn(


## Box-level agreement with fixations

Top-10%-mass connected components become boxes on both sides; then
pointing-game hits (argmax inside a human box), greedy IoU and fixation
mass covered. Same machinery for RoT arms, IG, occlusion, center and
noise.

In [5]:
from scipy.ndimage import label


def up(arr, size=128):
    return np.array(Image.fromarray(arr.astype(np.float32)).resize((size, size), Image.BILINEAR))

def boxes(sal, mass_q=10, min_frac=0.01):
    thr = np.quantile(sal.ravel(), 1 - mass_q / 100)
    lab, n = label((sal >= thr).astype(np.int32), structure=np.ones((3, 3), int))
    out = []
    for c in range(1, n + 1):
        ys, xs = np.nonzero(lab == c)
        if len(ys) / sal.size >= min_frac:
            out.append((xs.min(), ys.min(), xs.max(), ys.max()))
    return out

def iou(a, b):
    ix0, iy0, ix1, iy1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix1 - ix0 + 1) * max(0, iy1 - iy0 + 1)
    aa = (a[2] - a[0] + 1) * (a[3] - a[1] + 1)
    bb_ = (b[2] - b[0] + 1) * (b[3] - b[1] + 1)
    return inter / (aa + bb_ - inter)

def maps_of(exp, Xn):
    m = exp.model
    a, b = m.a.detach(), m.b.detach()
    out = []
    with torch.no_grad():
        for i, k in enumerate(P[idx]):
            x = torch.as_tensor(Xn[i:i + 1], device=m.device)
            r = m._respond(x)
            impk = a[int(k)][:, None, None] * (r[0] + b[int(k)][:, None, None])
            out.append(impk.abs().sum(0).cpu().numpy())
    return np.array(out)

mob_sals = np.array([up(m) for m in maps_of(exp_mob, Fm)])
pix_sals = np.array([up(m) for m in maps_of(exp_pix, Xp)])
fixes = F[idx]

def report(name, sals):
    pts, piou, cov = [], [], []
    for s, f in zip(sals, fixes):
        gb = boxes(f)
        peak = np.unravel_index(np.argmax(s), s.shape)
        pts.append(any(g[0] <= peak[1] <= g[2] and g[1] <= peak[0] <= g[3] for g in gb))
        pb = boxes(s)
        if gb and pb:
            piou.append(float(np.mean([max(iou(p, g) for g in gb) for p in pb])))
            m = np.zeros_like(f, dtype=bool)
            for x0, y0, x1, y1 in pb:
                m[y0:y1 + 1, x0:x1 + 1] = True
            cov.append(float(f[m].sum() / f.sum()))
    if piou and cov:
        print(f"{name}: pointing={np.mean(pts):.3f} P-IoU={np.mean(piou):.3f} coverage={np.mean(cov):.3f}", flush=True)
    else:
        print(f"{name}: pointing={np.mean(pts):.3f} (no matched boxes)", flush=True)

yy, xx = np.mgrid[0:128, 0:128]
center = (((xx - 64) ** 2 + (yy - 64) ** 2) <= 32 ** 2).astype(float)
rr = np.random.RandomState(0)
report("RoT-maps ", mob_sals)
report("RoT-pixel", pix_sals)
report("center   ", [center] * len(idx))
report("random   ", [rr.random((128, 128)) for _ in idx])

RoT-maps : pointing=0.353 P-IoU=0.197 coverage=0.298


RoT-pixel: pointing=0.220 P-IoU=0.078 coverage=0.166


center   : pointing=0.380 P-IoU=0.382 coverage=0.731


random   : pointing=0.093 (no matched boxes)


## IG and occlusion on a 20-image subset

Same box machinery; raw (unnormalised) maps so the comparison is fair.

In [6]:
from captum.attr import IntegratedGradients, Occlusion

sub = np.random.RandomState(0).choice(len(idx), 20, replace=False)
gi = idx[sub]
xb20 = torch.from_numpy(X[gi].transpose(0, 3, 1, 2).astype(np.float32) / 255.0)
tgt = torch.from_numpy(P[gi].astype(int))
ig = IntegratedGradients(bb)
ig_sals = np.array([ig.attribute(xb20[j:j + 1], target=int(tgt[j]),
                    baselines=torch.zeros_like(xb20[j:j + 1]),
                    n_steps=20)[0].detach().abs().sum(0).numpy() for j in range(20)])
occ = Occlusion(bb)
occ_sals = np.array([occ.attribute(xb20[j:j + 1], target=int(tgt[j]), strides=(3, 16, 16),
                     sliding_window_shapes=(3, 16, 16),
                     baselines=0.0)[0].detach().abs().sum(0).numpy() for j in range(20)])
report("IG       ", ig_sals)
report("occlusion", occ_sals)

IG       : pointing=0.300 P-IoU=0.239 coverage=0.378


occlusion: pointing=0.350 P-IoU=0.112 coverage=0.270


## Contact sheet: photo, RoT-maps boxes, human boxes

In [7]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from ruleofthumb import plot

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for r in range(4):
    j = int(gi[r])
    k = int(sub[r])
    axes[r, 0].imshow(X[j])
    axes[r, 0].set_title(f"photo {j} (box class {P[j]})")
    ax = axes[r, 1]
    ax.imshow(X[j])
    for x0, y0, x1, y1 in boxes(mob_sals[k]):
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="red", lw=2))
    ax.set_title("RoT-maps boxes")
    ax = axes[r, 2]
    ax.imshow(X[j])
    for x0, y0, x1, y1 in boxes(F[j]):
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="lime", lw=2))
    ax.set_title("human fixation boxes")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
plot.saliency(mob_sals[0] - mob_sals[0].mean(), image=X[idx[0]])
print("contact sheet + overlay rendered", flush=True)

contact sheet + overlay rendered


## Verdict

The maps arm tracks the box (agreement ~0.95+) while the pixel arm
collapses — fidelity tells you which arm to trust before any human
comparison. On boxes, RoT-maps lands at baseline parity (pointing ~0.35,
matching IG at ~0.30 and occlusion at ~0.35), well above the pixel arm
(~0.22) and noise (~0.09); the
static center prior still wins (fixations are central), which is reported,
not hidden. A faithful coarse distiller — never a gaze predictor.